<a href="https://colab.research.google.com/github/LeandroLDA/ColabNotebooks/blob/EmCriacao/TACTD_Avaliacao_Final_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="blue"> MBA em Ciência de Dados</font>
# <font color="blue">Técnicas Avançadas de Captura e Tratamento de Dados</font>

## <font color="blue">Avaliação Final - 2025</font>
**Luis Gustavo Nonato** e **Moacir Antonelli Ponti**<br>

**Cemeai - ICMC/USP São Carlos**

A avaliação vale 10 pontos e está dividida em duas partes, cada uma valendo 5.0 pontos.


<font color='red'>**ATENÇÃO:** Quando terminar a avaliação, você deve fazer um "upload" do notebook no _moodle_</font>.

## Parte 1

Os exercícios da Parte 1 fazem uso da coleção de documentos contida no pacote de dados do <font color='blue'> scikit </font> learn. A célula abaixo carrega a base de dados de documentos associados a quatro categorias diferentes. A célula também carrega os pacotes necessários para resolver os exercícios da Parte 1.

In [44]:
from sklearn.datasets import fetch_20newsgroups
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

nltk.download('punkt_tab')
nltk.download('stopwords')

category = ['sci.space', 'rec.autos', 'comp.graphics', 'misc.forsale']

newsgroups = fetch_20newsgroups(subset='train', categories=category)
docs = newsgroups.data # docs é uma lista onde cada elemento é um documento

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Questão 1 (2.5 pontos)
Construa um DataFrame <font color='blue'>pandas</font> onde as colunas correspondem aos documentos e as linhas às palavras que aparecem nos documentos. As palavras a serem inseridas nas linhas do DataFrame devem satisfazer:
- ser formada por dois ou mais caracteres do alfabeto (palavras com caracteres numéricos ou símbolos devem ser desconsideradas)
- não deve ser uma stopword (use a lista de stopwords do inglês)
- deve estar lexicamente normalizada

Cada palavra lexicamente normalizada deve aparecer apenas uma vez no DataFrame. Além disso, as entradas do DataFrame devem ser binárias, ou seja, se a palavra que aparece na linha $i$ está presente no documento representado pela coluna $j$, então a posição $(i,j)$ do DataFrame deve conter o valor 1, sendo 0 caso contrário.

Imprima as três palavras que aparecem em mais documentos. Em quantos documentos tais palavras aparecem?

In [54]:
#tokenizando as strings de docs
print("len de docs é: ",len(docs))
corpus = []

for i in docs:
  palavras = nltk.word_tokenize(i)
  corpus.extend(palavras)
print("len de corpus é: ",len(corpus))

len de docs é:  2356
len de corpus é:  708668


In [ ]:
lista_por_doc = []
todas_as_palavras = []



In [55]:
#removendo numericos e letras soltas
corpus = [w.lower() for w in corpus if w.isalpha() and len(w) != 1]
print("len de corpus é: ",len(corpus))

len de corpus é:  446875


In [56]:
# Removendo SW

stop_words = stopwords.words('english')

corpus = [w for w in corpus if w not in stop_words]
print("len de corpus é: ",len(corpus))

len de corpus é:  273980


In [58]:
# stemizando

qtde_antes_stem = len(corpus)
print(qtde_antes_stem)
corpus = [PorterStemmer().stem(w) for w in corpus]
corpus = list(set(corpus))
qtde_depois_stem = len(corpus)
print('Foram removidas ',qtde_antes_stem-qtde_depois_stem,' palavras')
print("len de corpus é: ",len(corpus))

18822
Foram removidas  134  palavras
len de corpus é:  18688


### Questão 2 (2.5 pontos)
Considere o DataFrame construído na questão 1.
- Remova do DataFrame as palavras que aparecem apenas em 1 documento e imprima quantas palavras foram removidas.
- Considerando cada coluna (documento) do DataFrame resultante da operação anterior com um ponto em um espaço de alta dimensão, calcule as direções principais dos pontos.
- Faça o gráfico da curva das variâncias explicadas e veja como elas tendem a zero.
- Qual o número mínimo de direções principais necessárias para capturar 80% da variância explicada?
- Projete os pontos nas duas direções principais com maior variância e faça um "scatter plot" dos pontos projetados.  

---
## Parte 2

### Questão 3  (2.5 pontos)

Carregue a base de dados California Housing conforme o código abaixo, e realize as seguintes tarefas:

1. Particione o conjunto aleatoriamente, sendo 10% reservado para teste
2. Remova outliers considerando as colunas em `check_outliers`, removendo as linhas com o critério de 3*IQR. Exiba quantos outliers foram removidos para cada coluna.
3. Normalize as colunas em `normalize` utilizando MinMaxScaler, para o intervalo entre 0 e 1.

Para as operações 2 e 3 acima, trate de forma adequada o processamento no conjunto de treinamento e teste.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import KBinsDiscretizer, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
import pandas as pd
import math

housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.columns)
numeric_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

check_outliers = ['AveRooms', 'AveBedrms']
normalize = ['AveRooms', 'AveBedrms', 'Latitude', 'Longitude']

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='object')


### Questão 4 (2,5 pontos)

Utilize o dataframe processado da questão 3.

1. Utilize o `KBinsDiscretizer` para discretizar em 5 bins utilizando `encode='ordinal'`, `strategy='quantile'` as variáveis contidas na lista `distretize`.
2. Treine um regressor utilizando LightGBM (LGBMRegressor) com seus hiperparametros padrão para predizer a coluna `MedHouseVal` a partir das demais colunas processadas anteriormente
3. Calcule o RMSE (Root Mean Squared Error) do conjunto de treinamento e de teste, e exiba esses erros.

Realize o processamento/tratamento adequado considerando as partições de treinamento e teste.


In [ ]:
discretize = ['MedInc', 'HouseAge', 'Population', 'AveOccup']

In [ ]:
# Mostra os primeiros 5 valores de target (categorias numéricas)
print("Primeiras 5 categorias (numéricas):", newsgroups.target[:5])

# Mostra os nomes das categorias
print("Nomes das categorias:", newsgroups.target_names)

# Para ver a categoria do primeiro documento (docs[0]):
categoria_primeiro_documento_numero = newsgroups.target[0]
nome_categoria_primeiro_documento = newsgroups.target_names[categoria_primeiro_documento_numero]
print(f"A categoria do primeiro documento é: {nome_categoria_primeiro_documento}")